# Diagnostics ported from cm1-out (PythonPlot)
Needs `_profiles.nc` (time-height, stability), `_surface.nc` (maps, PDFs) and `_fields.nc` (fluxes). Computations are in `diag.jl`; the plots are here.

In [ ]:
using Pkg; Pkg.activate(@__DIR__)
using Revise; includet("diag.jl")
using PythonPlot
rundir = "../run/ekamsat-p3-ctl0"

## Lowest-level maps: θ (liquid-ice), qv, u and v  (cm1-out `plot_out_allcases.jl`)
Breeze has no u10/v10 output; the lowest grid level u, v are used.

In [ ]:
n = last
θ = read_snapshot(rundir, "theta"; kind = "surface", n)
q = read_snapshot(rundir, "qv"; kind = "surface", n)
fig = figure(figsize = (9, 8))
for (k, (s, scale, label)) in enumerate(((θ, 1, "θ (K)"), (q, 1e3, "qv (g/kg)")))
    ax = fig.add_subplot(2, 1, k)
    pm = ax.pcolormesh(s.x ./ 1e3, s.y ./ 1e3, scale .* permutedims(s.data[:, :, 1]); cmap = "RdYlBu_r")
    fig.colorbar(pm; ax, label)
    ax.set_aspect("equal"); ax.set_ylabel("y (km)")
end
ax.set_xlabel("x (km)")
fig.suptitle("t = $(round(θ.t; digits = 2)) d"); fig.tight_layout()
fig

In [ ]:
u = read_snapshot(rundir, "u"; kind = "surface", n)
v = read_snapshot(rundir, "v"; kind = "surface", n)
fig = figure(figsize = (9, 8))
for (k, (s, label)) in enumerate(((u, "u (m/s)"), (v, "v (m/s)")))
    ax = fig.add_subplot(2, 1, k)
    m = maximum(abs, filter(isfinite, s.data))
    pm = ax.pcolormesh(s.x ./ 1e3, s.y ./ 1e3, permutedims(s.data[:, :, 1]); cmap = "RdYlBu_r", vmin = -m, vmax = m)
    fig.colorbar(pm; ax, label)
    ax.set_aspect("equal"); ax.set_ylabel("y (km)")
end
ax.set_xlabel("x (km)")
fig.suptitle("lowest grid level, t = $(round(u.t; digits = 2)) d"); fig.tight_layout()
fig

## Horizontal-mean resolved flux profiles  (cm1-out `plot_qflux.jl`, `plot_thflux.jl`)
Resolved ⟨w′a′⟩ only: the closure (subgrid) flux is not in the output.

In [ ]:
condensed = ["qv", "qcl", "qr"]          # total water; add the ice names of the run (qci qsn, or qi) as present
Fq = resolved_flux(rundir, condensed)
Fθ = resolved_flux(rundir, ["theta"])
fig = figure(figsize = (9, 5))
ax1 = fig.add_subplot(1, 2, 1); ax1.plot(Fq.flux, Fq.z ./ 1e3)
ax1.set_xlabel("⟨w′qₜ′⟩ (kg/kg m/s)"); ax1.set_ylabel("z (km)"); ax1.set_ylim(0, 12)
ax2 = fig.add_subplot(1, 2, 2); ax2.plot(Fθ.flux, Fθ.z ./ 1e3)
ax2.set_xlabel("⟨w′θ′⟩ (K m/s)"); ax2.set_ylim(0, 12)
fig.tight_layout(); fig

## Joint PDF of lowest-level θ and qv  (cm1-out `qth_sct.jl`)
The u*, q*, θ* PDFs need surface-flux outputs the model does not write yet.

In [ ]:
θe = range(extrema(filter(isfinite, θ.data))...; length = 80)
qe = range(extrema(filter(isfinite, q.data))...; length = 80)
h = zeros(79, 79)
for (a, b) in zip(vec(θ.data), vec(q.data))
    (isfinite(a) && isfinite(b)) || continue
    i = clamp(searchsortedlast(θe, a), 1, 79); j = clamp(searchsortedlast(qe, b), 1, 79)
    h[i, j] += 1
end
fig = figure(figsize = (6, 5)); ax = fig.add_subplot(1, 1, 1)
pm = ax.pcolormesh(collect(θe), 1e3 .* collect(qe), permutedims(h);
                   cmap = "RdYlBu_r", norm = PythonPlot.matplotlib.colors.SymLogNorm(linthresh = 1))
fig.colorbar(pm; ax, label = "count"); ax.set_xlabel("θ (K)"); ax.set_ylabel("qv (g/kg)")
fig

## Time-height qv, cloud and θ  (cm1-out `plot_diag.jl`)

In [ ]:
p = read_profiles(rundir)
fig = figure(figsize = (9, 5)); ax = fig.add_subplot(1, 1, 1)
cs = ax.contourf(p.t, p.z ./ 1e3, max.(p.qv, 1e-6); levels = 10 .^ range(-6, -1.5; step = 0.25),
                 norm = PythonPlot.matplotlib.colors.LogNorm(), cmap = "RdYlBu_r")
fig.colorbar(cs; ax, label = "qv (kg/kg)")
ax.contour(p.t, p.z ./ 1e3, p.qcl; levels = [1e-7, 1e-6], colors = "g")
ax.contour(p.t, p.z ./ 1e3, p.theta; levels = 280:2:500, colors = "k", linewidths = 0.5)
ax.set_ylim(0, 16); ax.set_xlabel("time (d)"); ax.set_ylabel("z (km)")
fig

## Moist N² and Richardson number
Pressure is hydrostatic from 1015 hPa (the files hold no pressure); N² uses θ_li.

In [ ]:
st = moist_stability(p)
fig = figure(figsize = (11, 5))
for (k, (A, label)) in enumerate(((st.N2moist, "moist N² (s⁻²)"), (log10.(max.(st.Rimoist, 1e-3)), "log₁₀ moist Ri")))
    ax = fig.add_subplot(1, 2, k)
    pm = ax.pcolormesh(st.t, st.z ./ 1e3, A; cmap = "RdBu_r", shading = "auto")
    fig.colorbar(pm; ax, label); ax.set_xlabel("time (d)"); ax.set_ylim(0, 16)
end
fig.tight_layout(); fig